In [1]:
import json
import os, glob

from bertopic import BERTopic
from transformers import AutoTokenizer, AutoModelForMaskedLM
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

import gensim.corpora as corpora
from gensim.models.coherencemodel import CoherenceModel

/Users/teamihajlov/Projects/CLIB_TopicVisualization/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_jsonl_files(folder_path):
    file_pattern = os.path.join(folder_path, "*.jsonl")
    file_paths = glob.glob(file_pattern)
    
    json_objects = []

    for file_path in file_paths:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                json_objects.append(json.loads(line))
    
    return json_objects

In [6]:
stopwords_path = "../final_stopwords_lat.txt"

with open(stopwords_path, 'r', encoding="utf-8") as f:
    stopwords = f.read().splitlines()

print(stopwords[-10:])
print(len(stopwords))
print(type(stopwords))

['nami', 'gdin', 'gđa', 'gđica', 'fra', 'frau', 'neje', 'bej', 'ča', 'han']
999
<class 'list'>


In [7]:
folder_path = "../cleaned_texts/"
json_data = load_jsonl_files(folder_path)

for obj in json_data[:5]:
    print(obj)

print(len(json_data))

{'text': 'zadruga ideal roman sunce rosa dolina zrak oblak magla raselina povesmo sredina zrak premagrevak šuma zelenilo trava dlan strana planina vrh ključ voda tišina vis bor jela tajac nastupanje vrućina ovca paš hlad lišće šibljika krava mesto koza drveće jezik izdanak usta zalogaj hladovina studenac trava čobanac dečko godina žubor studenac čistina nebo oči nedogled planina glas uzvik glava misao paponja koza moguu čuvarica glas neposlušnost jarac paponja paponjak dečko reč drugarica razmišljanje obrva znak nezadovoljstvo četvrt čas cveće borovina studenac misao dečko nebo učitelj duša zagonetka pamet svetac prilika munja duša glas đavo kozȃ paponja glava krava vreme šuma čobanica jarac noga prutić strana glava jarac paponja šala jahačica pesmica studenac voda torba jela prilika igra samoća reka devojče jarac rog devojka jarac rast stas vrata nebo oči obrva devojče usna poluosmejak oči prava devojče glava strana čuđenje jarac lik tromost posao ruka oči devojče cura prst zemlja vu

In [8]:
docs = []
for data in json_data:
    text = data["text"]
    docs.append(text)

print(docs[0])
print(len(docs))

zadruga ideal roman sunce rosa dolina zrak oblak magla raselina povesmo sredina zrak premagrevak šuma zelenilo trava dlan strana planina vrh ključ voda tišina vis bor jela tajac nastupanje vrućina ovca paš hlad lišće šibljika krava mesto koza drveće jezik izdanak usta zalogaj hladovina studenac trava čobanac dečko godina žubor studenac čistina nebo oči nedogled planina glas uzvik glava misao paponja koza moguu čuvarica glas neposlušnost jarac paponja paponjak dečko reč drugarica razmišljanje obrva znak nezadovoljstvo četvrt čas cveće borovina studenac misao dečko nebo učitelj duša zagonetka pamet svetac prilika munja duša glas đavo kozȃ paponja glava krava vreme šuma čobanica jarac noga prutić strana glava jarac paponja šala jahačica pesmica studenac voda torba jela prilika igra samoća reka devojče jarac rog devojka jarac rast stas vrata nebo oči obrva devojče usna poluosmejak oči prava devojče glava strana čuđenje jarac lik tromost posao ruka oči devojče cura prst zemlja vučenje noga

In [9]:
# mode_sbert = SentenceTransformer("distiluse-base-multilingual-cased-v2")

tokenizer_jerteh = AutoTokenizer.from_pretrained("jerteh/Jerteh-355")
model_jerteh = AutoModelForMaskedLM.from_pretrained("jerteh/Jerteh-355")

# tokenizer_bert = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# model_bert = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")

In [10]:
from sentence_transformers import SentenceTransformer

umap_model = UMAP(n_neighbors= 5, n_components= 5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size = 2, metric ='euclidean', cluster_selection_method = 'eom', prediction_data=True)
vectorizer_model = CountVectorizer(min_df = 3, max_df = 0.8, stop_words=stopwords, ngram_range = (1, 2))

In [11]:
topic_model = BERTopic(

  # language = 'multilingual',
  embedding_model=model_jerteh,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  min_topic_size = 5,

  top_n_words=20,
  verbose=True
)

topics, probs = topic_model.fit_transform(docs)

topics_bert = topic_model.get_topic_info()

2024-09-07 13:02:40,332 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]
2024-09-07 13:02:45,593 - BERTopic - Embedding - Completed ✓
2024-09-07 13:02:45,594 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
2024-09-07 13:02:48,043 - BERTopic - Dimensionality - Completed ✓
2024-09-07 13:02:48,044 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-09-07 13:02:48,048 - BERTopic - Cluster - Completed ✓
2024-09-07 13:02:48,049 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-09-07 13:02:49,120 - BERTopic - Representation - Completed ✓


In [12]:
topics_bert

,Topic,Count,Name,Representation,Representative_Docs
0,-1,9,-1_despot_kir_sultan_ćesar,"[despot, kir, sultan, ćesar, vezir, bula, knež...",[zadruga DIŠA roman DAVIDOVIĆ DIŠA soputnica ž...
1,0,19,0_hanum_fratar_robinja_gospoja,"[hanum, fratar, robinja, gospoja, hanuma, tatk...",[zadruga bakonja fra ĐAKOVANjE postrig izdanje...
2,1,10,1_arhimandrit_major_patrijarh_grof,"[arhimandrit, major, patrijarh, grof, nastojat...",[pero novac roman zavod put lekar dan leđa uho...
3,2,7,2_đakon_monah_kasta_frajla,"[đakon, monah, kasta, frajla, ogrlica, presedn...",[pripovetka varoš opisivanje geograf zemljovid...
4,3,6,3_aga_subaša_čorbadži_kahva,"[aga, subaša, čorbadži, kahva, hodža, tatko, l...",[pisac knjiga zona pripovetka prosveta zona gl...
5,4,6,4_nazaren_bukvar_gradina_tablica,"[nazaren, bukvar, gradina, tablica, tabla, cig...",[izdanje matica otac trava prst glava otac oči...
6,5,6,5_ćata_aga_gospa_front,"[ćata, aga, gospa, front, stanica, baterija, v...",[neimar roman istorija P UČITELj cena dinar de...
7,6,5,6_šator_arhimandrit_trpezarija_vojvoda knez,"[šator, arhimandrit, trpezarija, vojvoda knez,...",[subota jul godina lice roman put biblioteka u...
8,7,5,7_grofica_kneginjica_dragana_grof,"[grofica, kneginjica, dragana, grof, nana, ure...",[mar pripovetka naklada DIMITRIJEVIĆA ulica RA...
9,8,5,8_vezir_gospodin ministar_vranac_kavana,"[vezir, gospodin ministar, vranac, kavana, žan...",[majka pripovetka vek izdanje cena stovarište ...


In [13]:
topics_bert.to_csv('topics_bert_NOUN.csv')

In [14]:
topic_model.visualize_topics()

In [15]:
topic_model.visualize_barchart()

In [16]:
def make_sentence_transformer(
    model_name: str, max_seq_length: int = 512
) -> SentenceTransformer:
    # word_embedding_model = models.Transformer(model_name, max_seq_length=max_seq_length)
    # # Apply mean pooling to get one fixed sized sentence vector
    # pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(),
    #                             pooling_mode_cls_token=False,
    #                             pooling_mode_max_tokens=False,
    #                             pooling_mode_mean_tokens=True)
    # return SentenceTransformer(modules=[word_embedding_model, pooling_model])
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.model_max_length = max_seq_length  # Set the max length for the model
    tokenizer.padding_side = (
        "right"  # You can set "left" if you want to pad on the left side
    )
    # tokenizer.pad_token = tokenizer.eos_token  # Ensure the pad token is set
    model = SentenceTransformer(model_name)
    # Add the padding and truncation to the encode method
    model.tokenizer = tokenizer
    model.tokenizer_kwargs = {
        "padding": "max_length",
        "truncation": True,
        "max_length": max_seq_length,
        "return_tensors": "pt",  # Assuming you want PyTorch tensors as output
    }
    return model

In [17]:
jerteh_transformer = make_sentence_transformer("jerteh/Jerteh-355")

No sentence-transformers model found with name jerteh/Jerteh-355. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at jerteh/Jerteh-355 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
embeddings = []
for doc in docs:
    embedding = jerteh_transformer.encode(text)
    embeddings.append(embedding)

print(embeddings[0])
print(len(embeddings))

[-0.01702197 -0.02467978 -0.03548126 ... -0.02413623  0.03757919
  0.04753381]
108


In [19]:
reduced_embeddings = UMAP(n_neighbors=5, n_components=10, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

In [20]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 16/16 [00:00<00:00, 281.89it/s]


In [ ]:
cleaned_docs = topic_model._preprocess_text(docs)

vectorizer = topic_model.vectorizer_model
tokenizer = vectorizer.build_tokenizer()

words = vectorizer.get_feature_names_out()
tokens = [tokenizer(doc) for doc in cleaned_docs]
dictionary = corpora.Dictionary(tokens)
corpus = [dictionary.doc2bow(token) for token in tokens]
topic_words = [[words for words, _ in topic_model.get_topic(topic)] 
               for topic in range(len(set(topics))-1)]

coherence_model = CoherenceModel(topics=topic_words, 
                                 texts=tokens, 
                                 corpus=corpus,
                                 dictionary=dictionary, 
                                 coherence='c_v')
coherence = coherence_model.get_coherence()

In [ ]:
coherence

In [ ]:
top_words_per_topic = topics_bert['Representation'].tolist()

In [ ]:
def calculate_topic_diversity(topics, top_n=None):
    if top_n:
        # Trim the topics to the top_n words
        topics = [topic[:top_n] for topic in topics]
    
    unique_words = set()
    total_words = 0
    
    for topic in topics:
        unique_words.update(topic)
        total_words += len(topic)
    
    diversity = len(unique_words) / total_words if total_words > 0 else 0
    return diversity

In [ ]:
calculate_topic_diversity(top_words_per_topic, top_n = 10)